In [ ]:
import random
import json
import math
import numpy as np

from enum import IntEnum
from typing import Dict, Any, Sequence, Optional, Tuple, Set
from dataclasses import dataclass

from importnb import Notebook

with Notebook():
    # from LabTrajectory_RandomWalk import simulate_viewport_with_tiles
    from Labs.Trajectory_360Dataset import simulate_viewport_with_tiles, load_all_yaws_pitches
    from Labs.Trajectory_Pantelis import load_trajectories, simulate_viewport, map_tiles_viewport

import os
import sys

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

import Common.config as config
import Common.datatypes as datatypes
import Common.utils as utils

import importlib

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(utils)

<module 'Common.utils' from 'c:\\Users\\es25591\\Workspace\\CacheVideoPredict360\\Sources\\Common\\utils.py'>

In [ ]:
LayerType = datatypes.LayerType
WeibullParams = datatypes.WeibullParams
VideoCategory = datatypes.VideoCategory
ZipfSampler = datatypes.ZipfSampler

CATEGORY_WEIBULL = datatypes.CATEGORY_WEIBULL

In [ ]:
# -------------------------------
# Weibull parameterization
# -------------------------------
# Table I coefficients (alpha, beta, gamma) by category
# Source: Proactive Video Chunks Caching and Processing... (WCNC'19), Table I.  (α, β, γ)  [P-square ≈ 0.999]
# Note: keep category keys consistent in your dataset/taxonomy.

def _safe_pow(x: float, p: float) -> float:
    if x < 0:
        # guard tiny negatives caused by floating error for even/real powers
        x = 0.0
    return x ** p

def weibull_survival(c: float, params: WeibullParams) -> float:
    if c <= params.gamma:
        return 1.0
    z = (c - params.gamma) / params.beta
    return math.exp(-_safe_pow(z, params.alpha))

def prob_drop_between(c0: float, c1: float, params: WeibullParams) -> float:
    s0, s1 = weibull_survival(c0, params), weibull_survival(c1, params)
    p = max(0.0, s0 - s1)
    return min(p, 1.0)

def decide_continue_by_gop(current_gop: int,
                           total_gops: int,
                           category: VideoCategory,
                           rng: Optional[random.Random] = None) -> bool:

    params = CATEGORY_WEIBULL[category]
    c_g = current_gop / total_gops
    c_next = (current_gop + 1) / total_gops
    p_drop = prob_drop_between(c_g, c_next, params)

    return random.random() >= p_drop if rng else random.random() >= p_drop

In [ ]:
class UserRequestEvents:
    def __init__(
        self,
        n_nodes: int = 1,
        n_users: int = 1,
        step_size: float = 5.0,
        zipf_alpha: float = 1.0,
        n_videos: int = 100,
        n_gops: int = 60,
        n_layers: int = 1,
        n_tiles: int = 4,
        n: int = 4,
        m: int = 3,
        arrival_rate: float = 10.0,
        users_viewport_tiles: Optional[Sequence[Any]] = [],
        requested_videos: Optional[Sequence[Any]] = [],
        users_arrivals: Optional[Sequence[Any]] = [],
        cfg: Optional[config.Config] = None
    ):
        self.cfg = cfg or config.Config()

        self.user_gop_counter = [0 for _ in range(n_users)]
        self.user_total_gop_counter = [0] * n_users
        self.user_visited = [set() for _ in range(n_users)]
        
        # self.user_node_map = [random.randrange(n_nodes) for _ in range(n_users)]
        self.user_node_map = [i % n_nodes for i in range(n_users)]

        self.n_nodes = n_nodes
        self.n_users = n_users

        self.step_size = step_size
        self.zipf_alpha = zipf_alpha
        self.n_videos = n_videos
        self.n_gops = n_gops
        self.n_layers = n_layers
        self.n_tiles = n_tiles
        self.n = n
        self.m = m
        self.users_viewport_tiles = users_viewport_tiles
        self.requested_videos = requested_videos
        self.users_arrivals = users_arrivals
        self.arrival_rate = arrival_rate

        self.users_categories = {}
        self.trajectories = []

        self.reqs_arrived = []
    # ---------------------------------------
    # Internal Helpers
    # ---------------------------------------

    def _get_cache_status(
        self, 
        bitmaps: Optional[Any], 
        node_idx: int, 
        video_idx: int, 
        layer: int, 
        tile_idx: int, 
        gop: int
    ) -> bool:
        """Helper to safely check bitmap existence."""
        if bitmaps is None:
            return False
        
        if node_idx is None:
            return bitmaps[video_idx][layer][tile_idx][gop] == 1    

        return bitmaps[node_idx][video_idx][layer][tile_idx][gop] == 1

    def _create_tile_payload(
        self, 
        tile_idx: int, 
        layer: int, 
        size_total: float, 
        location_status: Tuple[bool, bool]
    ) -> Dict[str, Any]:
        """Generates the dictionary for a single tile request."""
        is_in_du, is_in_mec = location_status
        
        # Determine service location (One-hot logic)
        alpha_p_u = 1 if is_in_du else 0
        alpha_M_u = 1 if is_in_mec and not is_in_du else 0
        alpha_C_u = 1 if not is_in_du and not is_in_mec else 0

        return {
            "tile": tile_idx,
            "layer": layer,
            "size": size_total / self.cfg.n_tiles,
            "events": {
                "alpha_p_u": alpha_p_u,
                "alpha_M_u": alpha_M_u,
                "alpha_C_u": alpha_C_u,
                "beta_p_u": 0,
                "beta_M_u": 0
            }
        }
        
    def _start_new_video_session(self, user_idx: int):        
        """Resets a user state and assigns a new video from Zipf distribution."""
        self.user_gop_counter[user_idx] = 0
        
        # Sample new video
        if self.zipf_sampler:
            self.requested_videos[user_idx] = self.zipf_sampler() - 1
        
        # Simulate new viewport trajectory
        # Assumes simulate_viewport and self.trajectories are available
        self.users_viewport_tiles[user_idx] = simulate_viewport(
            self.requested_videos[user_idx],
            self.trajectories
        )

        # Update category
        if self.categories:
            video_idx = self.requested_videos[user_idx]
            self.users_categories[user_idx] = self.categories[video_idx % len(self.categories)]
    
    def _process_user_step(
        self, 
        user_idx: int, 
        du_bitmaps: Any, 
        mec_bitmap: Any
    ) -> Optional[Dict[str, Any]]:
        """Handles logic for a single user for the current step."""
        if self.user_is_done(user_idx):
            return None

        should_continue = True
        if self.user_gop_counter[user_idx] >= 1:            
            should_continue = decide_continue_by_gop(
                self.user_gop_counter[user_idx],
                self.cfg.n_gops,
                self.users_categories.get(user_idx)
            )

        if not should_continue or \
            self.user_gop_counter[user_idx] == len(self.users_viewport_tiles[user_idx]):
            self._start_new_video_session(user_idx)

        # 2. Check if user is active and generating requests
        has_arrived = self.users_arrivals[user_idx] <= self.step_count
        not_finished = self.user_total_gop_counter[user_idx] < self.cfg.user_session_length
        
        if has_arrived and not_finished:
            req = self.gen_request_for_user(user_idx, du_bitmaps, mec_bitmap)
            self.user_gop_counter[user_idx] += 1
            self.user_total_gop_counter[user_idx] += 1

            return req

        return None

    def _update_request_status(self, request: Dict[str, Any], du_bitmaps: Any, mec_bitmap: Any):
        """Updates the request's tile events based on current cache status."""
        user_idx = request["u"]
        node_idx = request["p"]
        video_idx = request["video"]
        gop = request["gop"]

        for tile in request["tiles"]:
            tile_idx = tile["tile"]
            layer = tile["layer"]

            is_in_du = self._get_cache_status(
                du_bitmaps, 
                node_idx, 
                video_idx, 
                layer, 
                tile_idx, 
                gop
            )
            
            is_in_mec = self._get_cache_status(
                mec_bitmap, 
                None, 
                video_idx, 
                layer, 
                tile_idx, 
                gop
            )

            # Update events based on actual cache status
            tile["events"]["alpha_p_u"] = 1 if is_in_du else 0
            tile["events"]["alpha_M_u"] = 1 if is_in_mec and not is_in_du else 0
            tile["events"]["alpha_C_u"] = 1 if not is_in_du and not is_in_mec else 0

    # ---------------------------------------
    # Main Logic
    # ---------------------------------------

    def gen_request_for_user(
        self, 
        user_idx: int, 
        du_bitmaps,
        mec_bitmap
    ) -> Dict[str, Any]:

        node_idx = self.user_node_map[user_idx]
        video_idx = self.requested_videos[user_idx]
        gop = self.user_gop_counter[user_idx]
        
        tiles_payload = []
        
        total_tiles = self.n * self.m
        
        for tile_idx in range(total_tiles):
            
            # Check DU (Edge)
            is_in_du = self._get_cache_status(
                du_bitmaps, 
                node_idx, 
                video_idx, 
                LayerType.BASE, 
                tile_idx, 
                gop
            )
            
            # Check MEC (Regional)
            is_in_mec = self._get_cache_status(
                mec_bitmap, 
                None, 
                video_idx, 
                LayerType.BASE, 
                tile_idx, 
                gop
            )

            tile_payload = self._create_tile_payload(
                tile_idx=tile_idx,
                layer=LayerType.BASE,
                size_total=self.cfg.bas_layer_size,
                location_status=(is_in_du, is_in_mec)
            )
            tiles_payload.append(tile_payload)

        # 2. Request Enhancement Layer (Layer 1)
        viewport_coords = self.users_viewport_tiles[user_idx][gop]
        
        for x, y in viewport_coords:
            tile_idx = y * self.n + x
            
             # Check DU (Edge)
            is_in_du = self._get_cache_status(
                du_bitmaps, 
                node_idx, 
                video_idx, 
                LayerType.ENHANCEMENT, 
                tile_idx, 
                gop
            )
            
            # Check MEC (Regional)
            is_in_mec = self._get_cache_status(
                mec_bitmap, 
                None, 
                video_idx, 
                LayerType.ENHANCEMENT, 
                tile_idx, 
                gop
            )

            tile_payload = self._create_tile_payload(
                tile_idx=tile_idx,
                layer=LayerType.ENHANCEMENT,
                size_total=self.cfg.enh_layer_size,
                location_status=(is_in_du, is_in_mec)
            )
            tiles_payload.append(tile_payload)

        viewport_flat = np.array(
            [y * self.n + x for x, y in viewport_coords], dtype=int
        )

        return {
            "u": user_idx,
            "p": node_idx, 
            "video": video_idx,
            "viewport": viewport_flat,
            "gop": gop,
            "tiles": tiles_payload,
            "base_req_init": (gop == 0),
            "T_chunk": self.cfg.T_chunk_default  
        }

    def step(self, du_bitmaps, mec_bitmap, user_visited: Optional[Set[int]] = None):
        """Advances the simulation by one time step for all users."""
        reqs = []

        for i in range(self.n_users):
            req = self._process_user_step(i, du_bitmaps, mec_bitmap)
            if req:
                reqs.append(req)

        self.step_count += 1
        return reqs

    def step_single_user(self, user_idx: int, du_bitmaps, mec_bitmap):
        """Advances simulation for a specific user (Intervention mode)."""
        return self._process_user_step(user_idx, du_bitmaps, mec_bitmap)

    def get_next_request(self, du_bitmaps, mec_bitmap):

        while len(self.reqs_arrived) == 0 and not self.all_users_done():
            self.reqs_arrived = self.step(du_bitmaps, mec_bitmap)

        request = self.reqs_arrived.pop(0)

        if request:
            self._update_request_status(request, du_bitmaps, mec_bitmap)

        return request

    def user_is_done(self, user_idx: int) -> bool:
        # return self.user_gop_counter[user_idx] >= self.cfg.n_gops
        return self.user_total_gop_counter[user_idx] >= self.cfg.user_session_length

    def all_users_done(self) -> bool:
        return all(self.user_is_done(uid) for uid in range(self.n_users))

    def sample_random_viewport(self, video: int) -> Sequence[Tuple[int, int]]:
        trajectories = self.trajectories[video % 10]
        trajectory = trajectories[np.random.randint(len(trajectories))]

        return map_tiles_viewport(trajectory[0])

    def get_user_gop(self, u_id: int) -> int:
        return self.user_gop_counter[u_id]

    def reset(self, **kwargs):
        self.step_count = 0
        self.user_gop_counter = [0] * self.n_users
        self.user_visited = [False] * self.n_users
        self.user_total_gop_counter = [0] * self.n_users

        self.reqs_arrived = []

        self.users_arrivals = utils.poisson_per_users(
            total_users=self.n_users,
            rate_per_minute=self.arrival_rate
        )
        self.users_arrivals = np.ones(self.n_users, dtype=int)
        # print(f"User arrivals (in steps): {self.users_arrivals}")
        
        self.zipf_sampler = ZipfSampler( 
            total_videos=self.n_videos, 
            alpha=self.zipf_alpha,
            seed=None
        )
        
        self.requested_videos = [
            self.zipf_sampler() - 1 for _ in range(self.n_users)
        ]

        self.categories = list(VideoCategory)

        self.users_categories = {
            u: self.categories[v % len(self.categories)] 
            for u, v in enumerate(self.requested_videos)
        }

        ### Load dataset based on Pantelis' traces  ###
        self.trajectories = load_trajectories(
            self.cfg.path_trajectories,
        )
        
        self.users_viewport_tiles = []
        for i in range(self.n_users):
            video = self.requested_videos[i]
            viewport_tiles = simulate_viewport(
                video, 
                self.trajectories
            )
            self.users_viewport_tiles.append(viewport_tiles)
        
        info = {
            "viewport_tiles": self.users_viewport_tiles,
            "requested_videos": self.requested_videos,
            "users_arrivals": self.users_arrivals,
            "users_requests": [],
            "user_request": None
        }
        
        return None, info

In [5]:
# Validation helpers
def validate_request_struct(req, num_tiles: int) -> bool:
    print(req)
    assert set([
        'gop','u','p','video','tiles'
    ]).issubset(req.keys()), 'Missing top-level keys'
    
    tiles = req['tiles']
    
    # Allow empty tiles for initial step (gop == 0), otherwise expect full base layer
    if len(tiles) > 0:
        assert len([
            t for t in tiles if t['layer'] == 0
        ]) == num_tiles, 'Base layer tiles count mismatch'
    
    for t in tiles:
        for k in ['tile','layer','size','events']:
            assert k in t, f'Missing tile key {k}'
        
        ev = t['events']
        
        for ek in ['alpha_p_u','alpha_M_u','alpha_C_u','beta_p_u','beta_M_u']:
            assert ek in ev, f'Missing event key {ek}'
    
    return True

In [6]:
if __name__ == '__main__':
    steps_to_run = 3
    n_users = 10
    step_size = 5.0
    zipf_alpha = 1.0
    n_videos = 100
    n_gops = 60
    n_layers = 2    # base + enhancement
    n = 4           # grid dimension (n x n)
    m = 3
    n_tiles = n * m # total tiles
    
    print('='*5, 'UserTileRequestEvents Test Harness', '='*5)
    print(f'Grid: {n}x{m} -> {n_tiles} tiles | Users: {n_users} | Layers: {n_layers}')
    print('Running steps...')

    user_env = UserRequestEvents(
        n=n,
        n_gops=n_gops,
        n_nodes=5,
        n_users=n_users,
        step_size=step_size,
        zipf_alpha=zipf_alpha,
        n_videos=n_videos,
        n_layers=n_layers,
        n_tiles=n_tiles
    )

    user_env.reset()
    user_env.users_arrivals[:] = 0

    cache_bitmap = np.zeros((
        user_env.n_videos, 
        user_env.n_layers,
        user_env.n_tiles, 
        user_env.n_gops
    ), dtype=np.int8)
    cache_bitmap[:, 0, :, :] = 1  # base layer cached for all videos

    all_requests = []
    for gop in range(steps_to_run):
        enh_layer_v0 = np.zeros(n_tiles, dtype=int)
        enh_layer_v1 = np.zeros(n_tiles, dtype=int)

        random_tile_indices = random.sample(range(n_tiles), 4)
        for idx in random_tile_indices:
            enh_layer_v0[idx] = 1

        random_tile_indices = random.sample(range(n_tiles), 4)
        for idx in random_tile_indices:
            enh_layer_v1[idx] = 1

        cache_bitmap[0, 1, :, gop] = enh_layer_v0  # video 0 enh layer
        cache_bitmap[1, 1, :, gop] = enh_layer_v1  # video 1 enh layer

        print(cache_bitmap[0, 1, :, gop])
        print(cache_bitmap[1, 1, :, gop])

        reqs = user_env.step(
            du_bitmaps=None,
            mec_bitmap=cache_bitmap
        )

        for r in reqs:
            validate_request_struct(r, n_tiles)
            enh_tiles = [t for t in r['tiles'] if t['layer'] == 1]
            print(f"  User {r['u']} Video {r['video']} GOP {r['gop']} EnhTiles={len(enh_tiles)}")

        all_requests.extend(reqs)

        # reqs = user_env.step(
        #     du_bitmaps=None,
        #     mec_bitmap=cache_bitmap
        # )

    print('\nSummary:')
    print(f'Total requests collected: {len(all_requests)}')
    base_sizes = set(t['size'] for r in all_requests for t in r['tiles'] if t['layer']==0)
    enh_sizes = set(t['size'] for r in all_requests for t in r['tiles'] if t['layer']==1)
    print(f'Base layer size values: {base_sizes}')
    print(f'Enh layer size values: {enh_sizes}')

    print('\nSample request (first user, first step):')
    # Print all requests in a readable format
    print(json.dumps(all_requests, indent=4))

===== UserTileRequestEvents Test Harness =====
Grid: 4x3 -> 12 tiles | Users: 10 | Layers: 2
Running steps...
[0 0 0 1 0 1 1 0 0 0 0 1]
[0 0 0 1 0 1 0 0 0 0 1 1]
{'gop': 0, 'u': 0, 'p': 0, 'video': 10, 'viewport': array([ 6,  7, 10, 11]), 'tiles': [{'tile': 0, 'layer': 0, 'size': 166666.66666666666, 'events': {'alpha_p_u': 0, 'alpha_M_u': 1, 'alpha_C_u': 0, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile': 1, 'layer': 0, 'size': 166666.66666666666, 'events': {'alpha_p_u': 0, 'alpha_M_u': 1, 'alpha_C_u': 0, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile': 2, 'layer': 0, 'size': 166666.66666666666, 'events': {'alpha_p_u': 0, 'alpha_M_u': 1, 'alpha_C_u': 0, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile': 3, 'layer': 0, 'size': 166666.66666666666, 'events': {'alpha_p_u': 0, 'alpha_M_u': 1, 'alpha_C_u': 0, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile': 4, 'layer': 0, 'size': 166666.66666666666, 'events': {'alpha_p_u': 0, 'alpha_M_u': 1, 'alpha_C_u': 0, 'beta_p_u': 0, 'beta_M_u': 0}}, {'tile': 5, 'layer': 0, 'size': 1666

TypeError: Object of type ndarray is not JSON serializable